# Course Outcome 1 (CO1) - Parking Lot Management System

## 1. Problem Statement
A parking lot needs a system to manage 100 parking slots. The system should track slot availability, allocate a free slot to an incoming vehicle, release the slot when the vehicle departs, calculate parking charges based on the duration of stay, and check if the parking area is full.

## 2. Divide into Parts/Modules
To solve this problem cleanly, the system is divided into the following functional parts:
1. **Core Operations Logic**: Functions to allocate and release slots (`allocate_slot`, `release_slot`).
2. **Computational Logic**: Functions to calculate parking fees and retrieve status metadata (`calculate_charges`, `get_occupancy_status`).
3. **Data Store & Input Setup**: Defines registry setups mapping integer slots 1 to 100 to vehicle identifiers.
4. **Driver Logic (Main)**: Interactive console menu loop to query occupancy, allocate/free slots, calculate prices, and handle validation boundaries.

## 3. Abstraction
Abstraction helps focus only on the essential data required to solve the problem and ignores unnecessary details.

### Needed Details (Essential Information):
* **Slot Number**: Integer key from 1 to 100 identifying specific spaces.
* **Occupancy Dictionary**: Data mapping of slots to vehicle plates.
* **Vehicle ID**: License plate string to register incoming/outgoing cars.
* **Duration**: Parking time used to compute billing fees.

### Useless Details (Irrelevant Information):
* **Vehicle Features**: Model, make, brand, paint color, or tire size.
* **Lot Structure**: Lane numbers, concrete marks, lighting, security gates, or CCTV.
* **Driver Identity**: Name, phone number, address, or payment cards.

## 4. Algorithm
The operations are performed as follows:
1. Initialize a dictionary `parking_lot` representing 100 slots, all mapped to `None` initially.
2. Loop to display options (1: Allocate Slot, 2: Release Slot, 3: View Available, 4: View Status, 5: Exit).
3. On allocation:
   - Search for the first key in 1-100 having `None` value.
   - If found, update value to vehicle identifier and output slot number.
   - If not found, report lot is full.
4. On release:
   - Prompt for slot number, retrieve vehicle identifier.
   - Clear slot value to `None`.
   - Prompt for duration, compute charge as `ceil(hours) * hourly_rate`, print billing invoice.
5. On view available:
   - Filter and display all slots with `None` values.
6. On occupancy status:
   - Print count of occupied and available slots. Flag if occupied >= 100.

## 5. Core Logic
Defines functions for slot allocation, release, and cost estimation.

In [1]:
def allocate_slot(parking_lot, vehicle_number):
    vehicle_number = vehicle_number.strip()
    if not vehicle_number:
        raise ValueError("Vehicle number cannot be empty.")
    for slot in range(1, 101):
        if parking_lot[slot] is None:
            parking_lot[slot] = vehicle_number
            return slot
    return None

def release_slot(parking_lot, slot_number):
    if not (1 <= slot_number <= 100):
        raise ValueError("Invalid slot number. Must be between 1 and 100.")
    vehicle_number = parking_lot[slot_number]
    if vehicle_number is None:
        return None
    parking_lot[slot_number] = None
    return vehicle_number

def calculate_charges(hours, rate=10.0):
    if hours <= 0:
        raise ValueError("Duration must be greater than zero.")
    import math
    return math.ceil(hours) * rate

def get_occupancy_status(parking_lot):
    occupied = [slot for slot, val in parking_lot.items() if val is not None]
    total = len(parking_lot)
    occupied_count = len(occupied)
    available_count = total - occupied_count
    return {
        "occupied_count": occupied_count,
        "available_count": available_count,
        "is_full": occupied_count >= total
    }


## 6. Data Store & Input Setup
Defines the initial state of the parking lot registry with some test vehicles.

In [2]:
parking_lot = {slot: None for slot in range(1, 101)}

parking_lot[5] = "KA-01-AA-9999"
parking_lot[12] = "DL-02-ZZ-1111"
parking_lot[50] = "MH-12-BB-2222"


## 7. Sample Edge Cases (Test Cases)
* **Test Case 1: Standard Allocation** -> Spot 1 is assigned to a new vehicle.
* **Test Case 2: Standard Release and Charges** -> Spot 1 is freed and fee is calculated based on hours.
* **Test Case 3: Slot Validation bounds** -> Allocating when full returns None; releasing an invalid index outputs ValueError.
* **Test Case 4: Fee rounding** -> Stay of 1.2 hours is billed as 2 hours ($20.00).

## 8. Driver Logic (Main)
Prompts the user to select either the Automated Demo or the Interactive Play mode.

In [3]:
operation_count = 0
max_operations = 30

try:
    while True:
        operation_count += 1
        if operation_count > max_operations:
            print("Reached maximum operation limit (30) to prevent kernel hang. Exiting.")
            break
            
        print("\n--- PARKING LOT MENU ---")
        print("1. Allocate Parking Slot")
        print("2. Release Parking Slot")
        print("3. View Available Slots")
        print("4. Show Occupancy Status")
        print("5. Exit")
        
        choice = input("Enter choice (1-5): ").strip()
        
        if choice == "1":
            vehicle_num = input("Enter Vehicle Plate Number: ").strip()
            try:
                allocated = allocate_slot(parking_lot, vehicle_num)
                if allocated is None:
                    print("Sorry, the parking lot is full.")
                else:
                    print(f"Slot {allocated} has been successfully allocated to {vehicle_num}.")
            except ValueError as e:
                print(f"Error: {e}")
                
        elif choice == "2":
            try:
                slot_no = int(input("Enter Slot Number to release (1-100): "))
                vehicle_num = release_slot(parking_lot, slot_no)
                if vehicle_num is None:
                    print(f"Slot {slot_no} is already vacant.")
                else:
                    hours = float(input("Enter duration parked (in hours): "))
                    fee = calculate_charges(hours)
                    print("-" * 45)
                    print("PARKING INVOICE")
                    print("-" * 45)
                    print(f"Released Slot:      {slot_no}")
                    print(f"Vehicle Number:     {vehicle_num}")
                    print(f"Duration:           {hours:.2f} hours")
                    print(f"Total Fee:          ${fee:.2f}")
                    print("-" * 45)
            except ValueError as e:
                print(f"Error: {e}")
                
        elif choice == "3":
            available = [slot for slot, val in parking_lot.items() if val is None]
            if not available:
                print("No slots available.")
            else:
                print("-" * 45)
                print("AVAILABLE PARKING SLOTS")
                print("-" * 45)
                chunk_size = 10
                for i in range(0, len(available), chunk_size):
                    chunk = available[i:i+chunk_size]
                    chunk_str = " ".join(f"{s:<3}" for s in chunk)
                    print(chunk_str)
                print("-" * 45)
                
        elif choice == "4":
            status = get_occupancy_status(parking_lot)
            print("-" * 45)
            print("PARKING LOT OCCUPANCY STATUS")
            print("-" * 45)
            print(f"Total Slots:      100")
            print(f"Occupied Slots:   {status['occupied_count']}")
            print(f"Available Slots:  {status['available_count']}")
            status_label = "FULL" if status['is_full'] else "Available (Not Full)"
            print(f"Status:           {status_label}")
            print("-" * 45)
            
        elif choice == "5":
            print("Exiting Parking Lot System. Goodbye!")
            break
        else:
            print("Invalid choice. Please select options 1-5.")
except (EOFError, Exception) as e:
    if "StdinNotImplementedError" in type(e).__name__ or isinstance(e, EOFError):
        print("Interactive session skipped in non-interactive environment.")
    else:
        raise e



--- PARKING LOT MENU ---
1. Allocate Parking Slot
2. Release Parking Slot
3. View Available Slots
4. Show Occupancy Status
5. Exit
Interactive session skipped in non-interactive environment.
